# UJIAN TENGAH SEMESTER — DATA ACQUISITION & MANAGEMENT
## Analisis Jumlah Kunjungan Wisatawan berdasarkan Jenis Wisatawan di Provinsi Nusa Tenggara Barat

**Nama:** Mikael Gunawan Antul  
**NIM:** 2501010318  
**Kelas:** F  
**Email kampus pribadi:** mikiantul@gmail.com

Notebook ini disusun mengikuti instruksi UTS: (1) pembacaan dataset terbuka dan estimasi storage, (2) lisensi, PII/quasi-identifier dan Data Ethics Checklist 9 butir, (3) akuisisi API minimal 30 baris, (4) pemeriksaan robots.txt dan scraping halaman statis yang diizinkan, (5) merge minimal dua sumber dengan `validate`, pemeriksaan duplikasi, dan penyimpanan metadata.

**Topik:** Pariwisata Nusa Tenggara Barat (NTB).  
**Dataset utama:** Jumlah Kunjungan Wisatawan berdasarkan Jenis Wisatawan, Satu Data NTB.

## A. Persiapan proyek dan folder

Folder `data/raw` menyimpan file sumber tanpa perubahan. Hasil normalisasi dan penggabungan disimpan terpisah pada `data/processed`. Jika notebook dijalankan di Google Colab, proyek dibuat di `/content/UTS-DAM-Pariwisata-NTB`.

In [ ]:
from pathlib import Path
import os, re, json, time, hashlib, platform
from datetime import datetime
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
from urllib.robotparser import RobotFileParser
from IPython.display import display

PROJECT = Path('/content/UTS-DAM-Pariwisata-NTB')
RAW = PROJECT / 'data' / 'raw'
PROCESSED = PROJECT / 'data' / 'processed'
META = PROJECT / 'metadata'
for folder in [RAW, PROCESSED, META]:
    folder.mkdir(parents=True, exist_ok=True)

# Isi dengan email kampus pribadi Anda sesuai instruksi dosen.
CAMPUS_EMAIL = 'GANTI_DENGAN_EMAIL_KAMPUS_PRIBADI'
USER_AGENT = f'UTS-DAM-Student-Project ({CAMPUS_EMAIL})'
print('Folder proyek:', PROJECT)
print('Email User-Agent sudah diisi:', CAMPUS_EMAIL != 'GANTI_DENGAN_EMAIL_KAMPUS_PRIBADI')

### A.1 Memasukkan dataset asli

File Excel yang kamu berikan sudah disertakan pada paket proyek di `data/raw`. Jika membuka notebook secara terpisah di Colab, upload file Excel tersebut ketika diminta.

In [ ]:
from google.colab import files
import shutil

dataset_path = RAW / 'dataset_kunjungan_wisatawan_ntb_asli.xlsx'
if not dataset_path.exists():
    print('Upload file XLSX dataset dari Satu Data NTB.')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError('File dataset belum diunggah.')
    uploaded_name = next(iter(uploaded))
    dataset_path.write_bytes(uploaded[uploaded_name])  # simpan byte asli tanpa edit
else:
    print('Dataset ditemukan:', dataset_path)

print('Ukuran file (byte):', dataset_path.stat().st_size)

## 1. Soal 1 — Membaca dan mengidentifikasi dataset

Dataset dibaca dari sheet `Worksheet`. Kolom `Jumlah (Orang)` disimpan sebagai teks pada file sumber karena angka menggunakan tanda koma pemisah ribuan, lalu dikonversi menjadi numerik hanya pada salinan analisis.

In [ ]:
# Baca Excel dengan sheet_name yang ditentukan dan tipe awal yang aman.
df_raw = pd.read_excel(
    dataset_path,
    sheet_name='Worksheet',
    dtype={'Kode provinsi': 'string', 'Jumlah (Orang)': 'string'}
)
print('Ukuran dataset:', df_raw.shape)
print('Kolom:', df_raw.columns.tolist())
display(df_raw.head())
display(df_raw.dtypes.to_frame('tipe_data_awal'))

In [ ]:
# Salinan analisis; file asli di data/raw tidak disentuh.
df = df_raw.copy()
df.columns = ['nomor','kode_provinsi','nama_provinsi','kategori_wisatawan',
              'jumlah_orang','tahun','bulan']

df['jumlah'] = pd.to_numeric(
    df['jumlah_orang'].astype(str).str.replace(',', '', regex=False),
    errors='coerce'
)
df['tahun'] = pd.to_numeric(df['tahun'], errors='coerce').astype('Int64')
df['kode_provinsi'] = pd.to_numeric(df['kode_provinsi'], errors='coerce').astype('Int64')
df['bulan'] = df['bulan'].astype(str).str.strip()

# Periksa ejaan kategori. Perbaikan hanya pada salinan analisis.
print('Kategori sebelum standardisasi:')
display(df['kategori_wisatawan'].value_counts(dropna=False))
df['kategori_wisatawan'] = df['kategori_wisatawan'].replace({
    'Wisatawan Mancaegara': 'Wisatawan Mancanegara'
})
print('Kategori setelah standardisasi:')
display(df['kategori_wisatawan'].value_counts(dropna=False))
display(df.head())

### 1.1 Jenis, cakupan, satuan, ukuran, dan kualitas data

Data berbentuk terstruktur karena tersusun sebagai tabel. Satuan `jumlah` adalah orang; cakupan wilayah adalah Provinsi Nusa Tenggara Barat; periode dan frekuensi aktual diperiksa dari isi file. Perhitungan storage mengikuti instruksi dosen: ukuran file × 12 bulan, ditambah cadangan 30%.

In [ ]:
file_bytes = dataset_path.stat().st_size
file_mb = file_bytes / (1024**2)
annual_storage_bytes = file_bytes * 12 * 1.30

print('Jenis data: Terstruktur (tabular)')
print('Jumlah baris:', len(df))
print('Jumlah kolom:', len(df.columns))
print('Kolom analisis:', df.columns.tolist())
print('Tipe data analisis:')
display(df.dtypes.to_frame('dtype'))
print('Wilayah:', ', '.join(df['nama_provinsi'].dropna().astype(str).unique()))
print('Tahun:', int(df['tahun'].min()), 's.d.', int(df['tahun'].max()))
print('Bulan unik:', sorted(df['bulan'].dropna().unique().tolist()))
print('Satuan jumlah: orang')
print(f'Ukuran file: {file_bytes:,} byte = {file_mb:.4f} MB')
print(f'Estimasi storage 1 tahun: {annual_storage_bytes:,.0f} byte = {annual_storage_bytes/(1024**2):.4f} MB')
print('Asumsi: satu file dengan ukuran sama dikumpulkan setiap bulan (12 bulan), lalu ditambah cadangan 30%.')
print('Nilai kosong per kolom:')
display(df.isna().sum().to_frame('jumlah_kosong'))
print('Duplikasi baris penuh:', int(df.duplicated().sum()))

## 2. Soal 2 — Lisensi, PII/quasi-identifier, dan Data Ethics Checklist

Metadata halaman sumber utama menunjukkan `License Not Specified`. Karena lisensi tidak disebutkan secara eksplisit, laporan tidak boleh menyebutnya sebagai CC-BY atau lisensi terbuka tanpa bukti. Dataset tampak berupa agregat kunjungan berdasarkan kategori, bulan, dan tahun; tetap periksa kolom aktual. Tahun, bulan, kategori, dan wilayah adalah atribut pengelompokan yang dapat menjadi quasi-identifier dalam konteks tertentu, tetapi di sini tidak menunjuk orang secara langsung.

In [ ]:
ethics = pd.DataFrame([
 ['1. Sumber dan tujuan','Satu Data NTB; digunakan untuk analisis akademik UTS','Sesuai','URL sumber utama dicatat pada provenance'],
 ['2. Lisensi dan ketentuan','Metadata sumber menampilkan License Not Specified','Perlu klarifikasi','Tidak mengklaim lisensi terbuka; atribusikan portal'],
 ['3. PII langsung','Tidak ditemukan nama/alamat/telepon individu pada struktur kolom yang diperiksa','Tidak teridentifikasi','Kolom: kode/nama provinsi, kategori, jumlah, tahun, bulan'],
 ['4. Quasi-identifier','Wilayah, tahun, bulan, kategori adalah atribut agregat','Risiko rendah','Tidak terdapat identitas individu dalam dataset agregat'],
 ['5. Minimisasi data','Analisis menggunakan variabel relevan: kategori, jumlah, tahun, bulan','Sesuai','File mentah dipisahkan dari salinan analisis'],
 ['6. Keamanan','Tidak menyimpan API key/token dalam notebook','Sesuai','API Open-Meteo yang digunakan tidak membutuhkan token'],
 ['7. Transparansi/provenance','URL, waktu akses, lisensi, nama file dan MD5 dicatat','Sesuai setelah sel provenance dijalankan','metadata/provenance.csv'],
 ['8. Kepatuhan API/scraping','Timeout, User-Agent email kampus, jeda, status code dan robots.txt diperiksa','Bersyarat','Bukti disimpan pada metadata dan output notebook'],
 ['9. Penyimpanan/publikasi','Raw tidak diubah; data turunan diberi sumber; tidak mengunggah rahasia','Sesuai','data/raw dan data/processed']
], columns=['butir','pemeriksaan','status','bukti_catatan'])
display(ethics)
ethics.to_csv(META/'data_ethics_checklist.csv', index=False)

## 3. Soal 3 — Akuisisi data melalui API (minimal 30 baris)

API yang dipilih adalah Open-Meteo Historical Weather API. Data cuaca harian titik Mataram, NTB, tahun 2024 digunakan sebagai variabel pendukung konteks pariwisata. Data cuaca tidak sama dengan data kunjungan wisatawan dan koordinat Mataram tidak mewakili seluruh kabupaten/kota NTB.

Request menggunakan timeout, User-Agent berisi email kampus pribadi, jeda, pemeriksaan status code, dan retry untuk status sementara.

In [ ]:
if CAMPUS_EMAIL == 'GANTI_DENGAN_EMAIL_KAMPUS_PRIBADI' or '@' not in CAMPUS_EMAIL:
    raise ValueError('Ganti CAMPUS_EMAIL dengan email kampus pribadi sebelum request.')

API_URL = 'https://archive-api.open-meteo.com/v1/archive'
api_params = {
    'latitude': -8.5833,
    'longitude': 116.1167,
    'start_date': '2024-01-01',
    'end_date': '2024-12-31',
    'daily': 'temperature_2m_mean,precipitation_sum,wind_speed_10m_max',
    'timezone': 'Asia/Makassar'
}
headers = {'User-Agent': USER_AGENT}

def request_with_retry(url, params=None, headers=None, timeout=30, retries=3):
    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, headers=headers, timeout=timeout)
            if response.status_code == 200:
                return response
            if response.status_code in (429, 500, 502, 503, 504):
                time.sleep((attempt + 1) * 2)
                continue
            response.raise_for_status()
        except requests.RequestException:
            if attempt == retries - 1:
                raise
            time.sleep((attempt + 1) * 2)
    raise RuntimeError('Request gagal setelah beberapa percobaan.')

time.sleep(1)
api_response = request_with_retry(API_URL, params=api_params, headers=headers)
api_json = api_response.json()
api_raw_path = RAW/'open_meteo_mataram_2024_raw.json'
api_raw_path.write_text(json.dumps(api_json, ensure_ascii=False, indent=2), encoding='utf-8')
print('Status HTTP:', api_response.status_code)
print('URL request:', api_response.url)
print('File JSON mentah:', api_raw_path)

In [ ]:
weather = pd.DataFrame(api_json['daily']).rename(columns={
    'time':'tanggal',
    'temperature_2m_mean':'suhu_rata_rata_c',
    'precipitation_sum':'curah_hujan_mm',
    'wind_speed_10m_max':'kecepatan_angin_maks_kmh'
})
weather['tanggal'] = pd.to_datetime(weather['tanggal'])
weather['tahun'] = weather['tanggal'].dt.year
weather['bulan_angka'] = weather['tanggal'].dt.month
if len(weather) < 30:
    raise ValueError(f'API menghasilkan {len(weather)} baris; perlu minimal 30.')
weather.to_csv(PROCESSED/'data_api_cuaca_harian.csv', index=False)
print('Jumlah baris API:', len(weather))
display(weather.head())

## 4. Soal 4 — Pemeriksaan robots.txt dan web scraping halaman statis

Target awal adalah halaman detail dataset resmi yang menjadi sumber dataset utama. Notebook memeriksa `robots.txt` lebih dahulu. Jika robots.txt tidak mengizinkan URL target, notebook tidak melakukan scraping dan mencatat hasil pemeriksaan. Jangan mengakali pembatasan; pilih halaman statis lain yang diizinkan jika target ditolak.

Jika diizinkan, BeautifulSoup mengambil metadata dan tabel/listing yang tersedia pada halaman tersebut. Request memakai timeout, User-Agent email kampus, status code, dan jeda.

In [ ]:
robots_status = None
robots_allowed = False
robots_text = ''
scrape_status = None
scrape_rows = 0
scrape_note = ''
try:
    time.sleep(1)
    robots_response = request_with_retry(ROBOTS_URL, headers=headers, timeout=25)
    robots_status = robots_response.status_code
    robots_text = robots_response.text
    (META/'robots_ntb.txt').write_text(robots_text, encoding='utf-8')
    robot_parser = RobotFileParser()
    robot_parser.parse(robots_text.splitlines())
    robots_allowed = robot_parser.can_fetch(USER_AGENT, SCRAPE_URL)
except Exception as exc:
    scrape_note = f'robots.txt tidak dapat diverifikasi ({exc}); scraping dihentikan demi kepatuhan.'
print('Robots.txt HTTP status:', robots_status)
print('Target diizinkan:', robots_allowed)
print('Target:', SCRAPE_URL)
print('Catatan:', scrape_note)

In [ ]:
if robots_allowed:
    time.sleep(2)
    page_response = request_with_retry(SCRAPE_URL, headers=headers, timeout=30)
    scrape_status = page_response.status_code
    (RAW/'halaman_satu_data_ntb_raw.html').write_text(page_response.text, encoding='utf-8')
    soup = BeautifulSoup(page_response.text, 'html.parser')

    # Ekstrak tabel jika tersedia; jika tidak, ekstrak daftar tautan dataset yang tampak.
    try:
        tables = pd.read_html(page_response.text)
    except ValueError:
        tables = []
    if tables:
        scraped = pd.concat(tables, ignore_index=True)
        scrape_note = f'Mengambil {len(tables)} tabel HTML.'
    else:
        records = []
        for a in soup.select('a[href]'):
            label = a.get_text(' ', strip=True)
            href = a.get('href')
            if label and href:
                records.append({'teks_tautan':label, 'url':requests.compat.urljoin(SCRAPE_URL, href)})
        scraped = pd.DataFrame(records).drop_duplicates()
        scrape_note = 'Tidak ada tabel HTML; mengekstrak listing tautan halaman.'
    if scraped.empty:
        scrape_note += ' Tidak ditemukan baris yang dapat diekstrak.'
    else:
        scraped.to_csv(RAW/'hasil_scraping_satu_data_ntb.csv', index=False)
        scrape_rows = len(scraped)
    display(scraped.head(10))
else:
    scrape_note = 'Scraping tidak dijalankan karena robots.txt tidak mengizinkan URL target. Pilih halaman statis lain yang diizinkan; jangan bypass robots.txt.'
    print(scrape_note)

scrape_evidence = pd.DataFrame([{
    'robots_url':ROBOTS_URL, 'robots_http_status':robots_status,
    'target_url':SCRAPE_URL, 'robots_allowed':robots_allowed,
    'scrape_http_status':scrape_status, 'jumlah_baris_hasil':scrape_rows,
    'catatan':scrape_note
}])
scrape_evidence.to_csv(META/'bukti_robots_scraping.csv', index=False)
display(scrape_evidence)

## 5. Soal 5 — Penggabungan minimal dua sumber, validate, dan pemeriksaan duplikasi

Dataset kunjungan wisatawan digabungkan dengan API cuaca. Data cuaca harian diringkas per bulan, lalu digabung menggunakan kunci bermakna `tahun` dan `bulan_angka`. Dataset wisatawan memiliki dua kategori per bulan, sedangkan cuaca memiliki satu baris per bulan; karena itu validasi relasi yang sesuai adalah `many_to_one`.

Penggabungan dibatasi pada 2024 karena data API yang diambil adalah tahun 2024. Jika bulan/tahun 2024 tidak tersedia pada file sumber, notebook akan memberi peringatan dan hasil merge kosong; jangan mengubah data sumber untuk memaksakan hasil.

In [ ]:
month_map = {'januari':1,'februari':2,'maret':3,'april':4,'mei':5,'juni':6,
             'juli':7,'agustus':8,'september':9,'oktober':10,'november':11,'desember':12}
analysis = df.copy()
analysis['bulan_angka'] = analysis['bulan'].astype(str).str.lower().map(month_map)
analysis['tahun'] = pd.to_numeric(analysis['tahun'], errors='coerce').astype('Int64')
visitor_2024 = analysis.loc[analysis['tahun'].eq(2024)].dropna(subset=['bulan_angka','jumlah']).copy()
visitor_2024['bulan_angka'] = visitor_2024['bulan_angka'].astype(int)

weather_monthly = weather.groupby(['tahun','bulan_angka'], as_index=False).agg(
    suhu_rata_rata_c=('suhu_rata_rata_c','mean'),
    curah_hujan_bulanan_mm=('curah_hujan_mm','sum'),
    kecepatan_angin_maks_kmh=('kecepatan_angin_maks_kmh','max')
)
print('Baris wisatawan tahun 2024:', len(visitor_2024))
print('Baris cuaca bulanan:', len(weather_monthly))
print('Kunci duplikat pada cuaca:', weather_monthly.duplicated(['tahun','bulan_angka']).sum())

if visitor_2024.empty:
    print('PERHATIAN: tidak ada record wisatawan tahun 2024 pada dataset. Tidak ada merge yang dapat dibuat.')
    merged = pd.DataFrame()
else:
    merged = visitor_2024.merge(
        weather_monthly,
        on=['tahun','bulan_angka'],
        how='left',
        validate='many_to_one',
        indicator=True
    )
    print('Status hasil merge:')
    display(merged['_merge'].value_counts(dropna=False))
    print('Jumlah baris sebelum merge:', len(visitor_2024))
    print('Jumlah baris setelah merge:', len(merged))
    print('Duplikasi baris penuh setelah merge:', int(merged.duplicated().sum()))
    assert len(merged) == len(visitor_2024), 'Jumlah baris berubah; periksa kunci merge.'
    merged.to_csv(PROCESSED/'hasil_gabungan_wisatawan_dan_cuaca_2024.csv', index=False)
    display(merged.head(10))

## 6. Provenance, URL, tanggal akses, lisensi, dan hash MD5

MD5 dihitung dari file mentah lokal untuk pemeriksaan integritas. Catat tanggal akses saat notebook dijalankan. Untuk API, lisensi dan atribusi harus diperiksa pada dokumentasi resmi Open-Meteo. Untuk dataset NTB, metadata yang terlihat menyatakan `License Not Specified`.

In [ ]:
def md5_file(path):
    digest = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

accessed = datetime.now().astimezone().isoformat(timespec='seconds')
provenance_rows = []

provenance_rows.append({
    'sumber':'Dataset kunjungan wisatawan NTB',
    'url':'https://data.ntbprov.go.id/dataset/9e270f88-a604-4640-a4f0-24c996122b92/show',
    'tanggal_akses':accessed,
    'lisensi':'License Not Specified pada metadata portal; perlu konfirmasi',
    'file':'data/raw/dataset_kunjungan_wisatawan_ntb_asli.xlsx',
    'md5':md5_file(dataset_path)
})
provenance_rows.append({
    'sumber':'Open-Meteo Historical Weather API',
    'url':api_response.url,
    'tanggal_akses':accessed,
    'lisensi':'CC BY 4.0; atribusi Open-Meteo wajib (https://open-meteo.com/)' ,
    'file':'data/raw/open_meteo_mataram_2024_raw.json',
    'md5':md5_file(api_raw_path)
})
html_path = RAW/'halaman_satu_data_ntb_raw.html'
if html_path.exists():
    provenance_rows.append({
        'sumber':'Halaman detail Satu Data NTB (scraping)',
        'url':SCRAPE_URL,
        'tanggal_akses':accessed,
        'lisensi':'License Not Specified pada portal; atribusikan sumber',
        'file':'data/raw/halaman_satu_data_ntb_raw.html',
        'md5':md5_file(html_path)
    })

provenance = pd.DataFrame(provenance_rows)
display(provenance)
provenance.to_csv(META/'provenance.csv', index=False)

## 7. Ringkasan hasil dan kesimpulan

Gunakan output yang benar-benar muncul setelah menjalankan notebook. Jangan menuliskan hasil API/scraping yang belum berhasil. Kesimpulan harus membedakan dataset kunjungan (NTB, data bulanan) dari cuaca (titik Mataram, data harian yang diringkas bulanan).

In [ ]:
print('RINGKASAN PROYEK')
print('Dataset utama:', df.shape[0], 'baris dan', df.shape[1], 'kolom')
print('Tahun tersedia:', df['tahun'].min(), '-', df['tahun'].max())
print('Jumlah data API harian:', len(weather))
print('Robots mengizinkan target scraping:', robots_allowed)
print('Jumlah baris scraping:', scrape_rows)
print('Jumlah baris merge:', len(merged))
print('File hasil dan metadata tersimpan di:', PROJECT)

## 8. Referensi

1. Satu Data NTB, *Jumlah Kunjungan Wisatawan berdasarkan Jenis Wisatawan*: https://data.ntbprov.go.id/dataset/9e270f88-a604-4640-a4f0-24c996122b92/show
2. Open-Meteo, *Historical Weather API*: https://open-meteo.com/en/docs/historical-weather-api
3. Satu Data NTB, `robots.txt`: https://data.ntbprov.go.id/robots.txt

## 9. Checklist sebelum pengumpulan

- [ ] Nama, NIM, dan email kampus sudah diisi.
- [ ] Semua sel notebook dijalankan tanpa error yang belum dijelaskan.
- [ ] Dataset asli tetap berada di `data/raw` tanpa diedit.
- [ ] Data API minimal 30 baris.
- [ ] Scraping hanya dijalankan jika robots.txt mengizinkan.
- [ ] Data Ethics Checklist 9 butir tersedia.
- [ ] Provenance memuat URL, tanggal akses, lisensi, nama file, dan MD5.
- [ ] Merge memakai `validate` dan pemeriksaan duplikasi.
- [ ] File hasil berada di `data/processed`.
- [ ] README dan seluruh folder diunggah ke GitHub.